# 50_00 - Phase 5 Overview, Portability Checks, and Vision Transformer Readiness

This notebook does **not** train a model.

It verifies that the vision-transformer pipeline is ready to run safely across devices and operating systems.

It is designed to be:

- Run All safe
- portable across Windows and Linux
- aligned with the shared data pipeline from earlier phases
- lightweight enough to rerun before starting any model notebook


In [1]:
from pathlib import Path
import json
import sys

import mlflow
import pandas as pd

NOTEBOOK_DIR = Path.cwd().expanduser().resolve()


def find_project_root(start: Path) -> Path:
    markers_all = ["src", "configs"]
    markers_any = [".git", "requirements.txt"]
    for candidate in [start, *start.parents]:
        if all((candidate / marker).exists() for marker in markers_all) and any((candidate / marker).exists() for marker in markers_any):
            return candidate
    raise RuntimeError(f"Could not locate project root from: {start}")


PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)
DATA_DIR = PROJECT_ROOT / "data"
CONFIGS_DIR = PROJECT_ROOT / "configs"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_METRICS_DIR = REPORTS_DIR / "metrics"
TRACKING_DIR = PROJECT_ROOT / "mlruns"
TRANSFORMS_CONFIG_PATH = CONFIGS_DIR / "transforms_v1.yaml"
SPLIT_DIR = DATA_DIR / "splits" / "split_v1"
TRAIN_CSV = SPLIT_DIR / "train.csv"
VAL_CSV = SPLIT_DIR / "val.csv"
TEST_CSV = SPLIT_DIR / "test.csv"
CLASSES_JSON = SPLIT_DIR / "classes.json"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT       :", PROJECT_ROOT)
print("REPORTS_METRICS_DIR:", REPORTS_METRICS_DIR)
print("TRACKING_DIR       :", TRACKING_DIR)


/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT       : /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server
REPORTS_METRICS_DIR: /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/reports/metrics
TRACKING_DIR       : /home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/mlruns


In [2]:
from src.data.dataset_loader import ImageDataset
from src.data.transforms import apply_size_overrides, get_eval_transforms, get_train_transforms, load_transforms_config
from src.models.vit import (
    build_model,
    count_parameters,
    get_model_spec,
    get_weights_name,
    list_available_models,
)

required_paths = {
    "train_csv": TRAIN_CSV,
    "val_csv": VAL_CSV,
    "test_csv": TEST_CSV,
    "classes_json": CLASSES_JSON,
    "transforms_config": TRANSFORMS_CONFIG_PATH,
}
missing = [name for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing required inputs for overview: {missing}")

REPORTS_METRICS_DIR.mkdir(parents=True, exist_ok=True)
TRACKING_DIR.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(TRACKING_DIR.as_uri())
mlflow.set_experiment("AnimalClassification")

with open(CLASSES_JSON, "r", encoding="utf-8") as f:
    classes_to_idx = json.load(f)

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

base_cfg = load_transforms_config(TRANSFORMS_CONFIG_PATH)
default_cfg = apply_size_overrides(base_cfg, image_size=224, resize_size=256)
train_tfm = get_train_transforms(default_cfg)
eval_tfm = get_eval_transforms(default_cfg)

train_ds = ImageDataset(
    split_csv=TRAIN_CSV,
    transform=train_tfm,
    classes_to_idx=classes_to_idx,
    project_root=PROJECT_ROOT,
    normalize_paths=True,
)
val_ds = ImageDataset(
    split_csv=VAL_CSV,
    transform=eval_tfm,
    classes_to_idx=classes_to_idx,
    project_root=PROJECT_ROOT,
    normalize_paths=True,
)

model_rows = []
for model_name in list_available_models().keys():
    spec = get_model_spec(model_name)
    model = build_model(model_name=model_name, num_classes=len(classes_to_idx), pretrained=False)
    model_rows.append(
        {
            "model_name": model_name,
            "weights_name": get_weights_name(model_name=model_name, pretrained=True),
            "parameter_count": int(count_parameters(model)),
            "image_size": int(spec.image_size),
            "resize_size": int(spec.resize_size),
        }
    )

summary = {
    "phase": "phase5_overview_summary",
    "status": "ready",
    "project_root": str(PROJECT_ROOT),
    "split_id": "split_v1",
    "transforms_config": str(TRANSFORMS_CONFIG_PATH),
    "train_size": int(len(train_df)),
    "val_size": int(len(val_df)),
    "test_size": int(len(test_df)),
    "dataset_lengths": {"train": len(train_ds), "val": len(val_ds)},
    "device_default": "cuda" if __import__("torch").cuda.is_available() else "cpu",
    "supported_models": model_rows,
}
summary_path = REPORTS_METRICS_DIR / "phase5_overview_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("[OK] Saved overview summary to:", summary_path)


/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/.venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


{
  "phase": "phase5_overview_summary",
  "status": "ready",
  "project_root": "/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server",
  "split_id": "split_v1",
  "transforms_config": "/home/temporaryuser4/Desktop/rufiz/~/AnimalClassification/AnimalClassification_server/configs/transforms_v1.yaml",
  "train_size": 50127,
  "val_size": 6266,
  "test_size": 6266,
  "dataset_lengths": {
    "train": 50127,
    "val": 6266
  },
  "device_default": "cuda",
  "supported_models": [
    {
      "model_name": "vit_b_16",
      "weights_name": "IMAGENET1K_V1",
      "parameter_count": 85800963,
      "image_size": 224,
      "resize_size": 256
    },
    {
      "model_name": "swin_t",
      "weights_name": "IMAGENET1K_V1",
      "parameter_count": 27521661,
      "image_size": 224,
      "resize_size": 232
    },
    {
      "model_name": "swin_v2_s",
      "weights_name": "IMAGENET1K_V1",
      "parameter_count": 48970749,
      "image_size": 256,
      "resize

In [3]:
with mlflow.start_run(run_name="phase5_overview_summary_validation"):
    mlflow.log_param("stage", "phase5_overview_summary")
    mlflow.log_param("split_id", "split_v1")
    mlflow.log_param("num_supported_models", len(summary["supported_models"]))
    mlflow.log_metric("train_size", summary["train_size"])
    mlflow.log_metric("val_size", summary["val_size"])
    mlflow.log_metric("test_size", summary["test_size"])
    mlflow.log_artifact(str(summary_path))

print("[OK] MLflow logging complete.")


[OK] MLflow logging complete.
